# An example to generate STM from dynamic estimation

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from depsi.LAMBDA import *
from depsi.dynamic_estimation import *

# Set input parameters

In [ ]:
# set wavelength of Sentinel-1 (C-band)
wavelength = 0.055465763 # m
init_len = 50
sigma_acc = 0.003 # mm/yr^2
L = 30/365

# Initial gauss for the sigma of the unknown parameters
sigma_offset = 0.001 #m
sigma_vel =  0.0001 #m/yr
sigma_h = 5 #m
sigma_ther = 0.00005 #m/°C

# the option for the LAMBDA method
# method == 1: ILS with shrinking search
# method == 2: Integer rounding
# method == 3: Integer bootstrapping
# method == 4: PAR
# method == 5: ILS with Ratio Test
method = 3

# Load the STM derived from PS selection

In [ ]:
stm = xr.open_zarr('output_data/stm_amsterdam_8p.zarr')
mother_epoch = '20150711'
mother_idx = np.where(stm.time == mother_epoch)[0][0]

dates = stm['dates'].to_numpy()
days = stm['days']
years = stm['years']
temp = stm['temperature']
stm

# Choose the reference point

In [ ]:
# Select the point with the smallest NMAD for initialization as the reference point
nmad_init = np.array(stm.nmad_init)
ref_pnt_idx= int(np.argmin(nmad_init))
ref_pnt_idx

# Define functions

In [ ]:
def polynomial_fitting(x, y, degree=1):
    """
    Perform polynomial fitting on the input data (x, y).

    Parameters:
    
    x : array_like
        Independent variable data points.
    y : array_like 
        Dependent variable data points.
    degree: int
        Degree of the polynomial fit (default is 1 for linear).

    Returns:
    y_fitted : ndarray
        Fitted y values for the given x based on the polynomial fit.
    poly_fn : np.poly1d
        Polynomial function that represents the fit.
    coeffs : ndarray 
        Coefficients of the fitted polynomial.
    """
    # Validate inputs
    if len(x) != len(y):
        raise ValueError("Input arrays 'x' and 'y' must have the same length.")
    if degree < 1:
        raise ValueError("Degree must be at least 1.")

    # Perform polynomial fitting
    coeffs = np.polyfit(x, y, degree)
    poly_fn = np.poly1d(coeffs)
    y_fitted = poly_fn(x)

    return y_fitted, poly_fn, coeffs



def NMAD_to_sigma_phase(nmad, method):

    """
    Converts Normalized Median Absolute Deviation (NMAD) to the sigma of phase observations.

    This function uses an empirical cubic approximation (and is not based on physics):
    sigma = a + b*nmad + c*nmad**2 + d*nmad**3

    Parameters:
    nmad : array_like
        Normalized Median Absolute Deviation value.
    method : {'mean', 'mean_2_sigma'}
        Method used to compute sigma.
    
    Returns:
    sigma: ndarray
        Estimated sigma value.
    """
    if method == 'mean':
        a, b, c, d = -0.0144869469, 2.00028682, -5.23271341, 21.1111801
    elif method == 'mean_2_sigma':
        a, b, c, d = 0.01907808, 1.2852969, 1.90052824, 11.60677721
    else:
        raise ValueError("Invalid method. Choose 'mean' or 'mean_2_sigma'.")

    sigma = a + b*nmad + c*nmad**2 + d*nmad**3
    
    return sigma

# Prepare matrices to store results

In [ ]:
phs_unw_rec_fm_stm = np.zeros((len(stm.space), len(stm.time)))
phs_est_rec_ft_stm = np.zeros((len(stm.space), len(stm.time)))
dis_rec_ft_stm = np.zeros((len(stm.space), len(stm.time)))
dis_est_rec_ft_stm = np.zeros((len(stm.space), len(stm.time)))
vel_rec_ft_stm = np.zeros((len(stm.space), len(stm.time)))
acc_rec_ft_stm = np.zeros((len(stm.space), len(stm.time)))
height_rec_ft_stm = np.zeros((len(stm.space), len(stm.time)))
ther_rec_ft_stm = np.zeros((len(stm.space), len(stm.time)))
phs_est_rec_fm_stm = np.zeros((len(stm.space), len(stm.time)))
dis_rec_fm_stm = np.zeros((len(stm.space), len(stm.time)))
dis_est_rec_fm_stm = np.zeros((len(stm.space), len(stm.time)))
vel_rec_fm_stm = np.zeros((len(stm.space), len(stm.time)))
acc_rec_fm_stm = np.zeros((len(stm.space), len(stm.time)))
height_rec_fm_stm = np.zeros((len(stm.space), len(stm.time)))
ther_rec_fm_stm = np.zeros((len(stm.space), len(stm.time)))
vel_lin_rec_fm_stm = np.zeros(len(stm.space))

vc_dd = np.zeros((len(stm.space), len(stm.time)))
vc_dv = np.zeros((len(stm.space), len(stm.time)))
vc_da = np.zeros((len(stm.space), len(stm.time)))
vc_dh = np.zeros((len(stm.space), len(stm.time)))
vc_dt = np.zeros((len(stm.space), len(stm.time)))
vc_vv = np.zeros((len(stm.space), len(stm.time)))
vc_va = np.zeros((len(stm.space), len(stm.time)))
vc_vh = np.zeros((len(stm.space), len(stm.time)))
vc_vt = np.zeros((len(stm.space), len(stm.time)))
vc_aa = np.zeros((len(stm.space), len(stm.time)))
vc_ah = np.zeros((len(stm.space), len(stm.time)))
vc_at = np.zeros((len(stm.space), len(stm.time)))
vc_hh = np.zeros((len(stm.space), len(stm.time)))
vc_ht = np.zeros((len(stm.space), len(stm.time)))
vc_tt = np.zeros((len(stm.space), len(stm.time)))

# Implement the dynamic estimation for each arc in a loop

In [ ]:
pnt_i_idx = ref_pnt_idx
for j in range(len(stm.space)):
    if j != pnt_i_idx:
        pnt_j_idx = j
        print(j)
        # Extract information of the two points of the arc
        sd_complex_i =  stm.isel(space = pnt_i_idx).sd_complex
        h2ph_i = stm.isel(space = pnt_i_idx).h2ph_values
        nmad_inc_i = stm.isel(space = pnt_i_idx).nmad_inc_stm

        sd_complex_j=  stm.isel(space = pnt_j_idx).sd_complex
        h2ph_j = stm.isel(space = pnt_j_idx).h2ph_values
        nmad_inc_j = stm.isel(space = pnt_j_idx).nmad_inc_stm

        # Get the sigma for the arc with the NMAD from the incremental time series
        sigma_nmad_inc_i = NMAD_to_sigma_phase(nmad_inc_i, 'mean_2_sigma')
        sigma_nmad_inc_j = NMAD_to_sigma_phase(nmad_inc_j, 'mean_2_sigma')
        sigma_nmad_inc_arc = np.sqrt(np.square(sigma_nmad_inc_i) + np.square(sigma_nmad_inc_j))

        # Compute DD phase for the arc
        sd_complex_conj_i = sd_complex_i.conj()
        dd_arc = sd_complex_j*sd_complex_conj_i

        # Compute 'mean' h2ph value for the arc (which we currently model as the average of the two time series)
        h2ph_arc = (h2ph_i + h2ph_j)/2
        h2ph_arc = h2ph_arc.to_numpy()

        # Get the wrapped phase
        phs_wrapped = np.angle(dd_arc)

        # Calculate the unknowns with the recursive solution
        x_hats_fm, Qx_hats_fm, y_hats_fm, y_hats_unw_fm, x_hats_ft, Qx_hats_ft, y_hats_ft, y_hats_unw_ft = init_rec_estimation(
            init_len, wavelength, phs_wrapped, sigma_nmad_inc_arc, years, h2ph_arc, temp, sigma_acc, L, sigma_offset, sigma_vel, sigma_h, sigma_ther, method)
        
        dis_fm, vel_fm, acc_fm, height_fm, temp_fm = x_hats_fm.T
        dis_ft, vel_ft, acc_ft, height_ft, temp_ft = x_hats_ft.T

        # calculate the sum of displacement and residual
        dis_rec_ft = (y_hats_unw_ft - y_hats_ft)*(-wavelength/(4*np.pi))+dis_ft
        dis_rec_fm = (y_hats_unw_fm - y_hats_fm)*(-wavelength/(4*np.pi))+dis_fm

        # calculate the constant valocity with a linear model
        y_fitted, poly_fn, coeffs = polynomial_fitting(years, dis_rec_fm, degree=1)
        vel_lin_fm_rec = coeffs[0]

        # store the variables
        phs_unw_rec_fm_stm[j,:] = y_hats_unw_fm
        phs_est_rec_ft_stm[j,:] = y_hats_ft
        dis_rec_ft_stm[j,:] = dis_rec_ft
        dis_est_rec_ft_stm[j,:] = dis_ft
        vel_rec_ft_stm[j,:] = vel_ft
        acc_rec_ft_stm[j,:] = acc_ft
        height_rec_ft_stm[j,:] = height_ft
        ther_rec_ft_stm[j,:] = temp_ft
        phs_est_rec_fm_stm[j,:] = y_hats_fm
        dis_rec_fm_stm[j,:] = dis_rec_fm
        dis_est_rec_fm_stm[j,:] = dis_fm
        vel_rec_fm_stm[j,:] = vel_fm
        acc_rec_fm_stm[j,:] = acc_fm
        height_rec_fm_stm[j,:] = height_fm
        ther_rec_fm_stm[j,:] = temp_fm
        vel_lin_rec_fm_stm[j] = vel_lin_fm_rec

        vc_dd[j,:] = Qx_hats_fm[:,0,0]
        vc_dv[j,:] = Qx_hats_fm[:,0,1]
        vc_da[j,:] = Qx_hats_fm[:,0,2]
        vc_dh[j,:] = Qx_hats_fm[:,0,3]
        vc_dt[j,:] = Qx_hats_fm[:,0,4]
        vc_vv[j,:] = Qx_hats_fm[:,1,1]
        vc_va[j,:] = Qx_hats_fm[:,1,2]
        vc_vh[j,:] = Qx_hats_fm[:,1,3]
        vc_vt[j,:] = Qx_hats_fm[:,1,4]
        vc_aa[j,:] = Qx_hats_fm[:,2,2]
        vc_ah[j,:] = Qx_hats_fm[:,2,3]
        vc_at[j,:] = Qx_hats_fm[:,2,4]
        vc_hh[j,:] = Qx_hats_fm[:,3,3]
        vc_ht[j,:] = Qx_hats_fm[:,3,4]
        vc_tt[j,:] = Qx_hats_fm[:,4,4]

# Save the results from the dynamic estimation to the STM

In [ ]:
stm['phs_unw_fm_rec'] = xr.DataArray(phs_unw_rec_fm_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['phs_est_ft_rec'] = xr.DataArray(phs_est_rec_ft_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['dis_ft_rec'] = xr.DataArray(dis_rec_ft_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['dis_est_ft_rec'] = xr.DataArray(dis_est_rec_ft_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vel_ft_rec'] = xr.DataArray(vel_rec_ft_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['acc_ft_rec'] = xr.DataArray(acc_rec_ft_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['height_ft_rec'] = xr.DataArray(height_rec_ft_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['ther_ft_rec'] = xr.DataArray(ther_rec_ft_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['phs_est_fm_rec'] = xr.DataArray(phs_est_rec_fm_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['dis_fm_rec'] = xr.DataArray(dis_rec_fm_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['dis_est_fm_rec'] = xr.DataArray(dis_est_rec_fm_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vel_fm_rec'] = xr.DataArray(vel_rec_fm_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['acc_fm_rec'] = xr.DataArray(acc_rec_fm_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['height_fm_rec'] = xr.DataArray(height_rec_fm_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['ther_fm_rec'] = xr.DataArray(ther_rec_fm_stm, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vel_lin_fm_rec'] = xr.DataArray(vel_lin_rec_fm_stm, dims=('space'), coords={'space': stm.space})

stm['vc_dd'] = xr.DataArray(vc_dd, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_dv'] = xr.DataArray(vc_dv, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_da'] = xr.DataArray(vc_da, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_dh'] = xr.DataArray(vc_dh, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_dt'] = xr.DataArray(vc_dt, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_vv'] = xr.DataArray(vc_vv, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_va'] = xr.DataArray(vc_va, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_vh'] = xr.DataArray(vc_vh, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_vt'] = xr.DataArray(vc_vt, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_aa'] = xr.DataArray(vc_aa, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_ah'] = xr.DataArray(vc_ah, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_at'] = xr.DataArray(vc_at, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_hh'] = xr.DataArray(vc_hh, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_ht'] = xr.DataArray(vc_ht, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})
stm['vc_tt'] = xr.DataArray(vc_tt, dims=('space','time'), coords={'space': stm.space, 'time': stm.time})

# Make a new STM for the results

In [ ]:
# make a new STM for the results, and drop other variables except the variables in the given list
stm_rec = stm.copy()
list_keep = ['phs_unw_fm_rec', 'phs_est_ft_rec', 'dis_ft_rec', 'dis_est_ft_rec', 'vel_ft_rec', 'acc_ft_rec', 'height_ft_rec', 'ther_ft_rec', 'phs_est_fm_rec',
             'dis_fm_rec', 'dis_est_fm_rec', 'vel_fm_rec', 'acc_fm_rec', 'height_fm_rec', 'ther_fm_rec', 'vel_lin_fm_rec',
             'vc_dd', 'vc_dv', 'vc_da', 'vc_dh', 'vc_dt', 'vc_vv', 'vc_va', 'vc_vh', 'vc_vt', 'vc_aa', 'vc_ah', 'vc_at', 'vc_hh', 'vc_ht', 'vc_tt']

list_drop = [var for var in list(stm_rec.data_vars.keys()) if var not in list_keep]
stm_rec = stm_rec.drop_vars(list_drop)

In [ ]:
# write the sigma and correlation length into the STM
sigma_acc_pre = np.full(len(stm.space), sigma_acc)
corr_length_pre = np.full(len(stm.space), int(L*365))
stm_rec['sigma_acc_pre'] = xr.DataArray(sigma_acc_pre, dims=('space'), coords={'space': stm.space})
stm_rec['corr_length_pre'] = xr.DataArray(corr_length_pre, dims=('space'), coords={'space': stm.space})
stm_rec

# Save the STM to a Zarr file

In [ ]:
# stm_rec.to_zarr('ams_dym_%sp.zarr' % len(stm.space), safe_chunks=False) # raise TypeError(f"Expected a BytesBytesCodec. Got {type(data)} instead.")
stm_rec.to_zarr('ams_dym_%sp.zarr' % len(stm.space), zarr_format=2, safe_chunks=False)

# Load the new STM

In [ ]:
stm = xr.open_zarr('ams_dym_8p.zarr')

# Plot the estimated results for a specific point

In [ ]:
pnt_idx = 0
y_hats_unw_fm = stm['phs_unw_fm_rec'][pnt_idx, :].values*(-wavelength/(4*np.pi))
y_hats_fm = stm['phs_est_fm_rec'][pnt_idx, :].values*(-wavelength/(4*np.pi))
dis_fm = stm['dis_fm_rec'][pnt_idx, :].values
dis_est_fm = stm['dis_est_fm_rec'][pnt_idx, :].values
vel_fm = stm['vel_fm_rec'][pnt_idx, :].values
acc_fm = stm['acc_fm_rec'][pnt_idx, :].values
height_fm = stm['height_fm_rec'][pnt_idx, :].values
ther_fm = stm['ther_fm_rec'][pnt_idx, :].values

In [ ]:
plt.figure(figsize = (15, 10))
ax = plt.subplot(3, 2, 1)
ax.axvline(dates[init_len], color='black', linestyle='--')
ax.plot(dates, y_hats_unw_fm*1000, '.', markersize = 5, color='black', alpha=0.9, label = 'Unwrapped')
ax.plot(dates, y_hats_fm*1000, color = 'red', linestyle='--',linewidth = 1.5, alpha = 0.95, label = 'Estimated') 
ax.set_ylabel(r'DD phase ($\varphi_{i,j}$) [mm]', size=11)
ax.legend(loc=4, prop={'size':10})
ax.text(0.02, 0.9, '(a)', fontsize = 20, transform = ax.transAxes)


ax = plt.subplot(3, 2, 2)
ax.axvline(dates[init_len], color='black', linestyle='--')
ax.plot(dates, dis_fm*1000, '.', markersize = 5, color='black', alpha=0.9, label = 'Unwrapped')
ax.plot(dates, dis_est_fm*1000, color = 'red', linewidth = 2, linestyle='--', label = 'Estimated')
ax.set_ylabel('Position ($D$) [mm]', size=10)
ax.legend(loc=4, prop={'size':10})
ax.text(0.02, 0.9, '(d)', fontsize = 20, transform = ax.transAxes)


ax = plt.subplot(3, 2, 4)
ax.axvline(dates[init_len], color='black', linestyle='--')
ax.plot(dates, vel_fm*1000, color='red', linestyle='--')
ax.set_ylabel('Instantaneous velocity ($v$) [mm/yr]', size=10)
ax.text(0.02, 0.9, '(e)', fontsize = 20, transform = ax.transAxes)


acc_fm_std = np.std(acc_fm)
ax = plt.subplot(3, 2, 6)
ax.axvline(dates[init_len], color='black', linestyle='--')
ax.plot(dates, acc_fm*1000, color='red',  linestyle='--')
ax.set_ylabel('Instantaneous acceleration ($a$) [mm/yr$^2$]', size=10)
ax.text(0.02, 0.9, '(f)', fontsize = 20, transform = ax.transAxes)
ax.set_xlabel('Time', size=12)


ax = plt.subplot(3, 2, 3)
ax.axvline(dates[init_len], color='black', linestyle='--')
ax.plot(dates, height_fm, color='red', linestyle='--')
ax.set_ylabel(r'Residual cross-range distance ($\Delta H$) [m]', size=10)
ax.text(0.02, 0.9, '(b)', fontsize = 20, transform = ax.transAxes)


ax = plt.subplot(3, 2, 5)
ax.axvline(dates[init_len], color='black', linestyle='--')
ax.plot(dates, temp_fm*1000, color='red', linestyle='--')
ax.set_ylabel(r'Thermal expansion ($\eta$) [mm/K]', size=10)
ax.text(0.02, 0.9, '(c)', fontsize = 20, transform = ax.transAxes)
ax.set_xlabel('Time', size=12)
plt.show()